# 05 — MAD Assembly (Na Events)

Deze stap bouwt `MAD_shortterm` en `MAD_longterm` met weather, calendar, location **en events** in één pipeline.


In [1]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.parent.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.project_config import get_project_paths
from src.phase1.assembly import (
    add_coverage_flags,
    build_mad_table,
    drop_admin_columns,
    export_mad_outputs,
    load_mad_inputs,
    merge_quality_report,
    normalize_time_columns,
    prepare_calendar_for_merge,
    prepare_location_for_merge,
)

paths = get_project_paths(PROJECT_ROOT)
print(f"ROOT      : {paths.root}")
print(f"DATA_INT  : {paths.data_intermediate}")
print(f"DATA_PROC : {paths.data_processed}")



ROOT      : /Users/emilevandevoorde/Documents/mechelen_parking
DATA_INT  : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate
DATA_PROC : /Users/emilevandevoorde/Documents/mechelen_parking/data_processed


In [2]:
inputs = load_mad_inputs(paths=paths, include_events=True)

df_st = inputs["shortterm"]
df_lt = inputs["longterm"]
df_w = inputs["weather"]
df_cal = inputs["calendar"]
df_loc = inputs["location"]
df_events = inputs.get("events")

print("Geladen bestanden:")
print(f"  shortterm : {df_st.shape}")
print(f"  longterm  : {df_lt.shape}")
print(f"  weather   : {df_w.shape}")
print(f"  calendar  : {df_cal.shape}")
print(f"  location  : {df_loc.shape}")
print(f"  events    : {None if df_events is None else df_events.shape}")



Geladen bestanden:
  shortterm : (284524, 21)
  longterm  : (46643, 20)
  weather   : (52632, 19)
  calendar  : (2922, 14)
  location  : (15, 6)
  events    : (2922, 12)


In [3]:
normalize_time_columns(df_st=df_st, df_lt=df_lt, df_w=df_w)
df_cal_clean = prepare_calendar_for_merge(df_cal)
df_loc_clean = prepare_location_for_merge(df_loc)

print(f"Locatietabel na filtering: {df_loc_clean.shape}")
print(f"Unieke parkings in loc: {df_loc_clean['parking_id'].nunique()}")



Locatietabel na filtering: (11, 5)
Unieke parkings in loc: 11


In [4]:
MAD_st = build_mad_table(
    base_df=df_st,
    weather_df=df_w,
    calendar_df=df_cal_clean,
    location_df=df_loc_clean,
    events_df=df_events,
)
MAD_lt = build_mad_table(
    base_df=df_lt,
    weather_df=df_w,
    calendar_df=df_cal_clean,
    location_df=df_loc_clean,
    events_df=df_events,
)

MAD_st = add_coverage_flags(MAD_st)
MAD_lt = add_coverage_flags(MAD_lt)

MAD_st = drop_admin_columns(MAD_st)
MAD_lt = drop_admin_columns(MAD_lt)

print(f"MAD shortterm: {MAD_st.shape}")
print(f"MAD longterm : {MAD_lt.shape}")



MAD shortterm: (284524, 63)
MAD longterm : (46643, 62)


In [5]:
merge_quality_report(MAD_st, "MAD_shortterm")
merge_quality_report(MAD_lt, "MAD_longterm")

print("\nEvent-kolommen aanwezig in shortterm:")
event_cols = [
    "is_event_day","is_football_day","is_festival_day","is_procession_day",
    "is_kermis_day","is_carnival_day","event_scale_max","n_concurrent_events",
    "data_confidence","event_names_combined","football_kickoff_hour"
]
print([c for c in event_cols if c in MAD_st.columns])



MERGE-KWALITEITSRAPPORT — MAD_shortterm
  rows                     : 284,524
  NaN occupancy_rate       : 0
  NaN parking_location_cat : 0
MERGE-KWALITEITSRAPPORT — MAD_longterm
  rows                     : 46,643
  NaN occupancy_rate       : 0
  NaN parking_location_cat : 0

Event-kolommen aanwezig in shortterm:
['is_event_day', 'is_football_day', 'is_festival_day', 'is_procession_day', 'is_kermis_day', 'is_carnival_day', 'event_scale_max', 'n_concurrent_events', 'data_confidence', 'event_names_combined', 'football_kickoff_hour']


In [6]:
outputs = export_mad_outputs(
    paths=paths,
    df_loc_clean=df_loc_clean,
    mad_st=MAD_st,
    mad_lt=MAD_lt,
)

print("Export voltooid:")
for k, v in outputs.items():
    print(f"  {k:<12}: {v}")



Export voltooid:
  location    : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate/parking_location_clean.parquet
  mad_shortterm: /Users/emilevandevoorde/Documents/mechelen_parking/data_processed/MAD_shortterm.parquet
  mad_longterm: /Users/emilevandevoorde/Documents/mechelen_parking/data_processed/MAD_longterm.parquet


In [7]:
MAD_st_disk = pd.read_parquet(outputs["mad_shortterm"])
MAD_lt_disk = pd.read_parquet(outputs["mad_longterm"])

print("=== Integriteitscheck ===")
print(f"MAD_st rows: {len(MAD_st_disk):,}")
print(f"MAD_lt rows: {len(MAD_lt_disk):,}")
print(f"PK dupes MAD_st: {int(MAD_st_disk.duplicated(subset=['parking_id','rounded_hour']).sum()):,}")
print(f"PK dupes MAD_lt: {int(MAD_lt_disk.duplicated(subset=['parking_id','rounded_hour']).sum()):,}")
print(f"Eventdagen MAD_st: {int(MAD_st_disk['is_event_day'].sum()) if 'is_event_day' in MAD_st_disk.columns else 'n/a'}")



=== Integriteitscheck ===
MAD_st rows: 284,524
MAD_lt rows: 46,643
PK dupes MAD_st: 0
PK dupes MAD_lt: 0
Eventdagen MAD_st: 35765
